In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
GTSDB_ROOT = PROJECT_ROOT / "FullIJCNN2013"
RUNS_DIR = PROJECT_ROOT / "runs"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
print("GTSDB exists:", GTSDB_ROOT.exists())

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import yaml

In [ ]:
RAW_ROOT = GTSDB_ROOT
GT_FILE = RAW_ROOT / "gt.txt"

YOLO_ROOT = DATA_DIR / "yolo" / "gtsdb"
IMAGES_OUT = YOLO_ROOT / "images"
LABELS_OUT = YOLO_ROOT / "labels"

print("GT exists:", GT_FILE.exists())
print("RAW exists:", RAW_ROOT.exists())
print("YOLO out:", YOLO_ROOT)

In [ ]:
df = pd.read_csv(GT_FILE, sep=";", header=None)
df.columns = ["filename", "left", "top", "right", "bottom", "class_id"]

print(df.head())
print("Rows:", len(df))
print("Unique images:", df["filename"].nunique())
print("Class IDs:", df["class_id"].nunique(), " | min:", df["class_id"].min(), "max:", df["class_id"].max())

In [ ]:
for split in ["train", "val", "test"]:
    (IMAGES_OUT / split).mkdir(parents=True, exist_ok=True)
    (LABELS_OUT / split).mkdir(parents=True, exist_ok=True)

print("Created folders under:", YOLO_ROOT)

In [ ]:
rng = np.random.default_rng(42)

images = df["filename"].unique()
rng.shuffle(images)

n = len(images)
train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_imgs = set(images[:train_end])
val_imgs   = set(images[train_end:val_end])
test_imgs  = set(images[val_end:])

def split_of(fname: str) -> str:
    if fname in train_imgs: return "train"
    if fname in val_imgs:   return "val"
    return "test"

df["split"] = df["filename"].apply(split_of)

print(df["split"].value_counts())
print("Images train/val/test:", len(train_imgs), len(val_imgs), len(test_imgs))

In [ ]:
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def to_yolo_row(left, top, right, bottom, img_w, img_h, class_id):
    # clamp to image bounds
    left   = clamp(int(left),   0, img_w - 1)
    right  = clamp(int(right),  0, img_w - 1)
    top    = clamp(int(top),    0, img_h - 1)
    bottom = clamp(int(bottom), 0, img_h - 1)

    # invalid boxes skip
    if right <= left or bottom <= top:
        return None

    x_center = ((left + right) / 2) / img_w
    y_center = ((top + bottom) / 2) / img_h
    w = (right - left) / img_w
    h = (bottom - top) / img_h

    return f"{int(class_id)} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

In [ ]:
grouped = df.groupby("filename")

missing_imgs = 0
skipped_boxes = 0
written_label_files = 0

for fname, g in grouped:
    img_path = RAW_ROOT / fname
    img = cv2.imread(str(img_path))
    if img is None:
        missing_imgs += 1
        continue

    img_h, img_w = img.shape[:2]
    split = g["split"].iloc[0]

    label_path = LABELS_OUT / split / (Path(fname).stem + ".txt")

    lines = []
    for _, row in g.iterrows():
        yolo_line = to_yolo_row(row.left, row.top, row.right, row.bottom, img_w, img_h, row.class_id)
        if yolo_line is None:
            skipped_boxes += 1
            continue
        lines.append(yolo_line)

    label_path.write_text(("\n".join(lines) + "\n") if lines else "", encoding="utf-8")
    written_label_files += 1

print("Label files written:", written_label_files)
print("Missing/unreadable images:", missing_imgs)
print("Skipped invalid boxes:", skipped_boxes)

In [ ]:
converted = 0
failed = 0

images_list = list(images)
total = len(images_list)

for i, fname in enumerate(images_list, start=1):
    img_path = RAW_ROOT / fname
    img = cv2.imread(str(img_path))
    if img is None:
        failed += 1
        continue

    split = split_of(fname)
    out_path = IMAGES_OUT / split / (Path(fname).stem + ".png")
    ok = cv2.imwrite(str(out_path), img)
    if ok:
        converted += 1
    else:
        failed += 1

    if i % 200 == 0 or i == total:
        print(f"Progress: {i}/{total} | converted={converted} | failed={failed}")

print("Converted images:", converted)
print("Failed images:", failed)

In [ ]:
class_ids = sorted(df["class_id"].unique())
max_id = max(class_ids)

# einfache Namen (später kann man echte Klassennamen ergänzen)
names = [f"class_{i}" for i in range(max_id + 1)]

data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": names
}

yaml_path = YOLO_ROOT / "data.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")

print("Wrote:", yaml_path)
print(yaml_path.read_text()[:400])

In [ ]:
# 1) gibt es train images?
train_imgs_out = list((IMAGES_OUT / "train").glob("*.png"))
val_imgs_out   = list((IMAGES_OUT / "val").glob("*.png"))
test_imgs_out  = list((IMAGES_OUT / "test").glob("*.png"))

print("images/train:", len(train_imgs_out))
print("images/val:", len(val_imgs_out))
print("images/test:", len(test_imgs_out))

# 2) label existiert für ein paar bilder?
for p in train_imgs_out[:5]:
    label = LABELS_OUT / "train" / (p.stem + ".txt")
    print(p.name, "-> label exists:", label.exists(), "| label size:", label.stat().st_size if label.exists() else None)

In [ ]:
import matplotlib.pyplot as plt

# nimm ein bild aus train
p = train_imgs_out[0]
img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)

label_file = LABELS_OUT / "train" / (p.stem + ".txt")
lines = label_file.read_text().strip().splitlines()

h, w = img.shape[:2]

for line in lines:
    cls, xc, yc, bw, bh = line.split()
    xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
    x1 = int((xc - bw/2) * w)
    y1 = int((yc - bh/2) * h)
    x2 = int((xc + bw/2) * w)
    y2 = int((yc + bh/2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)

plt.figure(figsize=(10,6))
plt.imshow(img)
plt.axis("off")
plt.title(f"Sample: {p.name} with YOLO labels")
plt.show()